<a href="https://colab.research.google.com/github/evinracher/3008410-intelligent-systems/blob/main/week7/exercise1/Explainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### LLM Explainability

We will use a technique called the "Logit Lens." This allows us to "peek" inside the model’s hidden layers to see how a thought evolves from a random guess in the early layers to a confident answer in the final layer.

In a standard LLM, we only see the final output. But the model has many layers (e.g., 12 layers in GPT-2). With the Logit Lens, we force the model to make a prediction at Layer 1, Layer 6, and Layer 12 to see its "internal brainstorm."

In [1]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# 1. Load model and tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

# 2. Use a very direct prompt
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")

# 3. Get the internal states
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
    hidden_states = outputs.hidden_states # List of layers
    final_layer_norm = model.transformer.ln_f # This is the "Filter" we need!

def get_interpretable_guesses(layer_index, k=2):
    # Get the raw hidden state
    h = hidden_states[layer_index][0, -1, :]

    # CRITICAL STEP: Apply the model's final normalization to the hidden state
    # This "cleans" the data so the head can read it properly
    h_cleaned = final_layer_norm(h)

    # Project to vocabulary
    logits = model.lm_head(h_cleaned)
    probs = torch.softmax(logits, dim=-1)

    top_probs, top_indices = torch.topk(probs, k)

    results = []
    for i in range(k):
        word = tokenizer.decode([top_indices[i]])
        # Clean up the visualization (replace spaces with an underscore)
        #word = word.replace(" ", "_")
        conf = top_probs[i].item()
        results.append(f"'{word}' ({conf:.1%})")

    return " | ".join(results)

print(f"Prompt: '{prompt} [???]'\n")
print(f"{'LAYER':<10} | {'TOP 2 INTERNAL GUESSES'}")
print("-" * 100)

# We check Layer 0 (Input), Layer 6 (Middle), and Layer 12 (Output)
for i in [0, 2, 6, 10]:
    label = "Input" if i == 0 else f"Layer {i}"
    guesses = get_interpretable_guesses(i)
    print(f"{label:<10} | {guesses}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Prompt: 'The capital of France is [???]'

LAYER      | TOP 2 INTERNAL GUESSES
----------------------------------------------------------------------------------------------------
Input      | ' destro' (53.3%) | ' mathemat' (28.0%)
Layer 2    | ' now' (27.1%) | ' not' (25.6%)
Layer 6    | ' now' (55.2%) | ' still' (10.9%)
Layer 10   | ' France' (62.1%) | ' Paris' (18.2%)


Observations

The "Blurry" Start: In the very first layer, the model's guess is often nonsensical (e.g., "the" or "a"). It hasn't "processed" the logic of the sentence yet; it's just looking at the most common words in English.

The "Brainstorming" Middle: By the middle layers (Layer 6), you might see the model start to guess words related to geography or countries. It has identified the context but hasn't reached the fact yet.

The "Clear" Conclusion: By the final layer (Layer 12), the model has refined the signals and usually outputs "Paris" with high confidence.


2023-2026 Research:

- Mechanistic Interpretability: This code shows that the "answer" is built incrementally. Researchers use this to find "knowledge neurons"—specific spots in specific layers where the fact "Paris = France" is actually stored.

- Hallucination Detection: If the confidence in Layer 12 is very low (e.g., only 10%), researchers know the model is likely about to hallucinate, even if it sounds confident in the text it eventually prints.

Exercise

- Do the same with other prompt in english "" and confirm the results.
  
- What happens when using a prompt in spanish ""?


In [5]:
# Exercise 1
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

prompt = "The largest planet in the Solar System is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

hidden_states = outputs.hidden_states
final_layer_norm = model.transformer.ln_f

def get_interpretable_guesses(layer_index, k=5):
    h = hidden_states[layer_index][0, -1, :]
    h_cleaned = final_layer_norm(h)
    logits = model.lm_head(h_cleaned)
    probs = torch.softmax(logits, dim=-1)

    top_probs, top_indices = torch.topk(probs, k)

    results = []
    for i in range(k):
        word = tokenizer.decode([top_indices[i]]).strip()
        results.append((word, top_probs[i].item()))
    return results

for layer in [1, 6, 12]:
    print(f"\nLayer {layer}")
    for word, prob in get_interpretable_guesses(layer, k=5):
        print(f"{word!r}: {prob:.4f}")

with torch.no_grad():
    next_token_logits = outputs.logits[0, -1, :]
    next_token_id = torch.argmax(next_token_logits).item()
    next_token = tokenizer.decode([next_token_id]).strip()

print("Predicted next token:", next_token)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Layer 1
'not': 0.2717
'also': 0.1115
'now': 0.0942
'still': 0.0526
'a': 0.0346

Layer 6
'now': 0.3608
'probably': 0.1782
'still': 0.1478
'currently': 0.0632
'not': 0.0475

Layer 12
'the': 0.0431
',': 0.0265
'a': 0.0243
'in': 0.0190
'and': 0.0138
Predicted next token: about


In [7]:
# Exercise 2
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

prompt = "La capital de Francia es"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

hidden_states = outputs.hidden_states
final_layer_norm = model.transformer.ln_f

def get_interpretable_guesses(layer_index, k=5):
    h = hidden_states[layer_index][0, -1, :]
    h_cleaned = final_layer_norm(h)
    logits = model.lm_head(h_cleaned)
    probs = torch.softmax(logits, dim=-1)

    top_probs, top_indices = torch.topk(probs, k)

    results = []
    for i in range(k):
        token = tokenizer.decode([top_indices[i]]).strip()
        results.append((token, top_probs[i].item()))
    return results

print(f"Prompt: {prompt}")

for layer in [1, 6, 12]:
    print(f"\nLayer {layer}")
    for token, prob in get_interpretable_guesses(layer, k=5):
        print(f"{token!r}: {prob:.4f}")

with torch.no_grad():
    final_logits = outputs.logits[0, -1, :]
    final_probs = torch.softmax(final_logits, dim=-1)
    top_probs, top_indices = torch.topk(final_probs, 10)

print("\nFinal prediction (top 10 next tokens):")
for i in range(10):
    token = tokenizer.decode([top_indices[i]]).strip()
    print(f"{token!r}: {top_probs[i].item():.4f}")

predicted_token = tokenizer.decode([top_indices[0]]).strip()
print("\nMost likely next token:", predicted_token)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prompt: La capital de Francia es

Layer 1
'es': 0.9977
'ces': 0.0004
'port': 0.0002
'que': 0.0001
'ci': 0.0001

Layer 6
'ci': 0.2591
'es': 0.1912
'lev': 0.0691
'de': 0.0319
'la': 0.0315

Layer 12
',': 0.0244
'': 0.0182
'.': 0.0156
'the': 0.0152
'a': 0.0134

Final prediction (top 10 next tokens):
'la': 0.1553
'pa': 0.0967
'que': 0.0465
'el': 0.0220
'un': 0.0196
'se': 0.0179
'de': 0.0117
'est': 0.0093
'pe': 0.0091
'l': 0.0088

Most likely next token: la


R/= Como el modelo fue entrenado en inglés, no se tienen las relaciones semánticas para prediccir correctamente la siguiente parte de la oración, por lo que el resultado no final no tiene sentido semántica ni gramaticalmente. Con un modelo multi idioma o entrenado en español, la predicción sí funcionaría.